In [ ]:
%matplotlib inline

In [ ]:
# Import libraries here

import numpy as np
import pandas as pd
import itertools
import os
import sys
import scipy.optimize as sco

In [ ]:
current_dir = os.path.abspath('')
project_root = os.path.abspath(os.path.join(current_dir, '../../'))

if project_root not in sys.path:
    sys.path.append(project_root)

os.chdir(project_root)

print(f"Working directory set to: {os.getcwd()}")

In [ ]:
# Import project modules here

import importlib
from src.portfolio_simulation_utils import portfolio_simulation as p_sim
from src.data_pipeline_utils import data_fetching_handling as data_pipe
importlib.reload(p_sim)
importlib.reload(data_pipe)

## Harry Markowitz Efficient Frontier ##

Let us now repeat the exercise with the same stocks, however using more robust mathematical modeling skills. While I am building this project, I am very curious if applying the mathematical concepts more carefully will lead to a result, which is similar enough to the practical approach I have utilized in the past. 

The first step is the use the exact same tickers and rebuild our returns dataframe, very similar to the practical approach

In [ ]:
tickers = ['AAPL', 'NVDA', 'MSFT', 'JNJ', 'BAC', 'VZ', 'WMT', 'UPS', 'PFE', 'JPM']

In [ ]:
returns_df =  data_pipe.build_returns_df(tickers)

### 1. Define the initial inputs  ###

Some of the steps below should already be familiar to what we did with the monte carlo simulation and the mathematical approach. We would take the mean returns of the stocks, we will define the covariance matrix, and construct a portfolio with equal weights of the stocks. Through matrix multiplication we can define the expected portfolio return.These are the fundamental inputs for the analytical approach.Expected Returns Vector ($\mu$): Calculate the annualized mean of the daily returns for each asset.Covariance Matrix ($\Sigma$): Calculate the annualized covariance of the daily returns between all asset pairs.

In [ ]:
trading_days = 252
risk_free_rate = 0.03
mean_returns = returns_df.mean() * trading_days
cov_matrix = returns_df.cov() * trading_days

In [ ]:
print(mean_returns)

In [ ]:
print(cov_matrix)

### 2. Define the Linear Algebra  ###

Modern Portfolio Theory evaluates portfolios using matrix multiplication. You must define functions to compute portfolio performance given a weight vector ($w$).

Portfolio Return: $E(R_p) = w^T \mu$

Portfolio Variance: $\sigma_p^2 = w^T \Sigma w$

Let us define the starting weights vector by stating an equal weight of each stock in the portfolio for a starting point

In [ ]:
number_of_stocks = len(tickers)
weights = np.array([1 / number_of_stocks] * number_of_stocks)

In [ ]:
returns = np.dot(weights, mean_returns)
std_dev = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
print(returns, std_dev)

In [ ]:
def negative_sharpe_ratio(weights, mean_returns, cov_matrix, risk_free_rate=0.03):
    p_ret = np.dot(weights, mean_returns)
    p_std = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return -(p_ret - risk_free_rate) / p_std

In [ ]:
args = (mean_returns, cov_matrix)

# Constraint: sum of weights equals 1
constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})

# Bounds: weights between 0 and 1
bounds = tuple((0, 1) for asset in range(number_of_stocks))

# Initial guess (equal distribution)
initial_guess = number_of_stocks * [1. / number_of_stocks]

### 3. Run the computer algorithm for solving quadratic optimization  ###

We stopped short of doing this manually in notebook 1_2. This mathematics becomes very complicated and above my knowledgelevel. However, as promised we we will run a computer algorithm that does that for us. I have chosen the optimization module of the **scipy library** in python from the available options. In the variable **bounds** above, we added the crucial constraint that we do not allow short-selling. 

In [ ]:
optimal_portfolio = sco.minimize(
    negative_sharpe_ratio, 
    initial_guess, 
    args=args,
    method='SLSQP', 
    bounds=bounds, 
    constraints=constraints
)

optimal_weights = optimal_portfolio.x
optimal_return = np.sum(mean_returns * optimal_weights)
optimal_variance = np.dot(optimal_weights.T, np.dot(cov_matrix, optimal_weights))
optimal_sharpe_ratio = optimal_return / np.sqrt(optimal_variance)

print(f"Expected Return: {optimal_return:.4f}")
print(f"Variance: {optimal_variance:.4f}")

In [ ]:
optimal_portfolio = list(zip(tickers, optimal_weights))
simulated_portfolio = [] #copy-paste here the outcome of notebook 1_1 in order to do sanity check
print(optimal_portfolio)

### 4. Sanity Check ###

We can take the optimal portfolio using the algorithmic solution from above in a list and the optimal portfolio produced by the portfolio simulation done with Monte Carlo methods. As said, the great the number of sim runs is, the closer the result is. Unfortunately my machine takes nearly 20 minutes to do 1 000 000 runs and I haven't experimented with a greater number of runs. 

In [ ]:
list_1 = optimal_portfolio
list_2 = simulated_portfolio
for i in range(len(tickers)):
    print(f"Ticker {list_1[i][0]} -> Monte Carlo {list_1[i][1] * 100:.2f} - Math algorithm {list_2[i][1] * 100:.2f}"
    f"\nDifference {(list_1[i][1] * 100 - list_2[i][1] * 100):.2f}")

## Combinations of portfolios, which are subsets of the 10-ticker portfolio

Let's experiment with Combinations to make the study more exciting. It is going to be informative to find all possible portfolios of five stocks based on the ten stocks we have chosen. 

To find all possible combinations of 5 assets from a pool of 10, you use the binomial coefficient (combinations). The formula is:
$$\binom{n}{k} = \frac{n!}{k!(n-k)!}$$

If we run the formula, we find out that there are exactly 252 unique ways to select a 5-ticker portfolio from your list of 10.

We use the cool itertools library in Python to make it instantenous in applying this mathematical calculation

In [ ]:
portfolio_combinations = list(itertools.combinations(tickers, 5))

print(f"Total portfolios generated: {len(portfolio_combinations)}")
print(portfolio_combinations[0])

### Let's optimize all 252 portfolios we received ###

Below, we are going to replicate the portfolio optimization technique we applied above.

In [ ]:
number_of_stocks = 5
trading_days = 252
portfolio_metrics = {}
counter = 1

for portfolio in portfolio_combinations:

    current_weights = np.array([1 / number_of_stocks] * number_of_stocks)
    tickers = list(portfolio)
    returns_df =  data_pipe.build_returns_df(tickers)
    
    current_mean_returns = returns_df.mean() * trading_days
    current_cov_matrix = returns_df.cov() * trading_days

    returns, std_dev = calc_portfolio_performance(current_weights, current_mean_returns,current_cov_matrix)

    args = (current_mean_returns, current_cov_matrix)
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for asset in range(number_of_stocks))
    initial_guess = number_of_stocks * [1. / number_of_stocks]

    current_optimal_portfolio = sco.minimize(
    negative_sharpe_ratio, 
    initial_guess, 
    args=args,
    method='SLSQP', 
    bounds=bounds, 
    constraints=constraints
    )

    current_optimal_weights = current_optimal_portfolio.x
    current_return = np.sum(current_mean_returns * current_optimal_weights)
    current_variance = np.dot(current_optimal_weights.T, np.dot(current_cov_matrix, current_optimal_weights))
    current_sharpe = current_return / np.sqrt(current_variance)
    
    portfolio_metrics[f"Portfolio {counter}"] = {'Tickers': tickers,
                                  'Weights': current_optimal_weights, 
                                  'Return': current_return, 
                                  'Variance': current_variance,
                                  'Variance Deviation': current_variance - optimal_variance,
                                    "Sharpe Ratio": current_sharpe,
                                "Sharpe Deviation": optimal_sharpe_ratio - current_sharpe,
                                 }

    counter += 1

print(portfolio_metrics)

### Let's observe the results of this experiment."

We can see that the majority of the portfolios constructed by a subset of 5 stocks from the originally selected stocks exhibit worse risk characteristics than a portfolio constructed of all selected tickers. Yet, we can find 5-ticker portfolios, which are possible constructed using stocks with lower variance of their returns, i.e., less risky. However, when we compare the utilization of risk to achieve a unit of return, we observe that the optimized portfolio using all selected tickers is expected to be more efficient.  

While individual 5-stock subsets can achieve lower absolute variance by selecting low-volatility assets, the 10-stock universe consistently provides a superior Risk-Adjusted Return. The 10-stock (in our case) optimal portfolio maintains a Sharpe Ratio that is, on average, 0.20 to 0.60 points higher than 5-stock combinations, proving that a broader asset base allows for a more "efficient" efficient frontier.

In [ ]:
for portfolio, metric in portfolio_metrics.items():
    print(f"{portfolio} -> Variance Deviation: {metric['Variance Deviation'] * 100:.2f}, Sharpe Ratio Deviation: {metric['Sharpe Deviation']:.2f}")

## Conclusion of Part I – Portfolio Optimization

In the first sequence of notebooks — **1_1, 1_2, and 1_3** — we gradually built an understanding of what portfolio optimization means and why diversification creates measurable economic value.

We began with an intuitive and practical approach: generating large numbers of random portfolios through Monte Carlo simulation and observing how diversification reduces risk for a given level of expected return. Increasing the number of simulations demonstrated an important computational insight: brute-force experimentation can approximate optimal solutions surprisingly well when sufficient computing power is available.

We then moved from intuition to theory by revisiting the original framework established by **Harry Markowitz (1952)**. Modern Portfolio Theory formalizes diversification using the mathematical tools of **linear algebra, probability, and calculus**. Portfolio returns can be represented as vector operations, while portfolio risk emerges from a quadratic form involving the covariance matrix of asset returns. The resulting optimization problem reveals the curvature of the risk–return relationship and defines the **efficient frontier**.

However, the mathematics quickly becomes complex. Even with a relatively small number of assets, solving systems of quadratic expressions analytically can become cumbersome. This naturally motivates the use of **algorithmic approaches**, where Python and modern numerical libraries allow us to efficiently compute and visualize the efficient frontier.

What emerges from this exploration is an important realization: mathematics is not merely theoretical abstraction. Linear algebra, probability distributions, and optimization algorithms directly influence how trillions of dollars of capital are allocated in global financial markets.

While diversification improves the **risk–return tradeoff**, it does not eliminate risk entirely. This observation motivates the next step in our analysis.

In the second sequence of notebooks — **2_1 and 2_2** — we extend the same mathematical framework to **derivative instruments**, specifically options. Using the Black–Scholes model and the concept of option Greeks, we will show how derivatives introduce **curvature into payoff functions**, allowing investors to reshape the behavior of financial assets and even entire portfolios through hedging strategies.